### 3.9.19. Gauss-Newton Method

$$
J(x^k)^{\mathsf T}J(x^k)d^k = -J(x^k)^{\mathsf T}g(x^k),
\qquad
x^{k+1}=x^k+\alpha^k d^k.
$$


**Explanation:**

Gauss-Newton is a Newton-like method specialized to nonlinear least squares. Instead of forming the full Hessian of $\frac{1}{2}\|g(x)\|^2$, it linearizes the residual vector $g$ and solves the resulting linear least squares problem for a correction step. This replaces the Hessian by $J(x^k)^{\mathsf T}J(x^k)$, which is [positive definite](../../01_Foundations/01_Linear_Algebra/07_Theoretical_Linear_Algebra/17_positive_definite_matrix.ipynb) when the [Jacobian](../../01_Foundations/01_Linear_Algebra/08_Coordinate_Transformations/10_jacobian_matrix.ipynb) has full column rank. That rank condition connects the method to [rank-nullity](../../01_Foundations/01_Linear_Algebra/03_Matrix/12_matrix_rank_nullity.ipynb) and to [positive definite matrices](../../01_Foundations/01_Linear_Algebra/07_Theoretical_Linear_Algebra/17_positive_definite_matrix.ipynb).

The matrix $J(x^k)$ is the [Jacobian matrix](../../01_Foundations/01_Linear_Algebra/08_Coordinate_Transformations/10_jacobian_matrix.ipynb) of residual derivatives at the current iterate. The linear system is the normal-equation approximation for the Gauss-Newton direction $d^k$, and the second formula applies that direction with stepsize $\alpha^k$.

The approximation is especially good when the residuals are small near the solution or when the residual functions are nearly linear. If the residuals are large or the model is strongly nonlinear, the dropped second-order terms can matter, and damping or line search may be needed. Gauss-Newton is therefore both a practical algorithm and a conceptual link between linear least squares and Newton's method.

**Intuition:**

At each iteration, Gauss-Newton asks what parameter correction best fits the linearized residual equation. The code cell computes one such correction for an exponential model. The output is not meant to finish the optimization; it shows how one local linear least squares subproblem produces the next candidate point.


**Numerical Example:**

For the model $m(t;a,b)=ae^{bt}$, use the current parameter estimate $(a,b)=(0.8,0.9)$ and data
$$
(t,y)=(0,1.0),\ (1,2.7),\ (2,7.4).
$$

The model values are approximately
$$
(0.8e^0,\ 0.8e^{0.9},\ 0.8e^{1.8})=(0.8000,1.9677,4.8397),
$$
so the residual vector $g=m-y$ is
$$
g=\begin{bmatrix}-0.2000\\-0.7323\\-2.5603\end{bmatrix}.
$$

The Jacobian rows are
$$
\left[\frac{\partial m}{\partial a},\frac{\partial m}{\partial b}\right]
=\left[e^{bt},\ ate^{bt}\right],
$$
which gives
$$
J\approx
\begin{bmatrix}
1.0000&0.0000\\
2.4596&1.9677\\
6.0496&9.6794
\end{bmatrix}.
$$

The Gauss-Newton normal equations are
$$
J^{\mathsf T}Jd=-J^{\mathsf T}g,
\qquad
\begin{bmatrix}43.6479&63.3969\\63.3969&97.5633\end{bmatrix}d
=\begin{bmatrix}17.4900\\26.2231\end{bmatrix}.
$$

Solving gives
$$
d\approx\begin{bmatrix}0.1836\\0.1495\end{bmatrix},
\qquad
(a,b)+d\approx(0.9836,1.0495).
$$


In [1]:
import sympy as sp

p_1, p_2 = sp.symbols("p_1 p_2")
parameters = sp.Matrix([p_1, p_2])
sample_points = [0, 1, 2]
observations = [1, sp.Rational(27, 10), sp.Rational(37, 5)]
residual = sp.Matrix([p_1 * sp.exp(p_2 * point) - observation
                      for point, observation in zip(sample_points, observations)])
jacobian = residual.jacobian(parameters)

current_parameters = sp.Matrix([sp.Rational(4, 5), sp.Rational(9, 10)])
substitution = {p_1: current_parameters[0], p_2: current_parameters[1]}
residual_value = residual.subs(substitution)
jacobian_value = jacobian.subs(substitution)
step = (jacobian_value.T * jacobian_value).solve(-jacobian_value.T * residual_value)
updated_parameters = current_parameters + step

print("jacobian ="); sp.pprint(jacobian)
print("updated parameters ="); sp.pprint(sp.N(updated_parameters.T, 6))

jacobian =
⎡  1        0     ⎤
⎢                 ⎥
⎢  p₂         p₂  ⎥
⎢ ℯ       p₁⋅ℯ    ⎥
⎢                 ⎥
⎢ 2⋅p₂        2⋅p₂⎥
⎣ℯ      2⋅p₁⋅ℯ    ⎦
updated parameters =
[0.983574  1.04949]


**Equivalent `casadi` implementation:**

In [2]:
import casadi as ca

parameters = ca.SX.sym("p", 2)
sample_points = ca.DM([0, 1, 2])
observations = ca.DM([1, 2.7, 7.4])
model_values = parameters[0] * ca.exp(parameters[1] * sample_points)
residual = model_values - observations
jacobian = ca.jacobian(residual, parameters)

residual_function = ca.Function("r", [parameters], [residual])
jacobian_function = ca.Function("J", [parameters], [jacobian])

current_parameters = ca.DM([0.8, 0.9])
residual_value = residual_function(current_parameters)
jacobian_value = jacobian_function(current_parameters)
normal_matrix = jacobian_value.T @ jacobian_value
right_side = -jacobian_value.T @ residual_value
step = ca.solve(normal_matrix, right_side)
updated_parameters = current_parameters + step

print("current_parameters =", current_parameters)
print("residual_value =", residual_value)
print("normal_matrix =", normal_matrix)
print("right_side =", right_side)
print("step =", step)
print("updated_parameters =", updated_parameters)


current_parameters = [0.8, 0.9]
residual_value = [-0.2, -0.732318, -2.56028]
normal_matrix = 
[[43.6479, 63.3969], 
 [63.3969, 97.5633]]
right_side = [17.49, 26.2231]
step = [0.183574, 0.149493]
updated_parameters = [0.983574, 1.04949]


**References:**

[📘 Bertsekas, D. P. (1999). *Nonlinear Programming* (2nd ed.), Chapter 1 "Unconstrained Optimization". Athena Scientific.](https://www.athenasc.com/nonlinbook.html)

---

[⬅️ Previous: Least Squares Problem Models](./18_least_squares_problem_models.ipynb) | [Next: Incremental Gauss-Newton and Filtering ➡️](./20_incremental_gauss_newton_and_filtering.ipynb)

---
